## Análises e Insights

As queries abaixo devem ser executadas após o `dbt run`, lendo o `dw_elt` já populado.

### 1. Instalando as dependências

In [ ]:
%pip install psycopg2-binary sqlalchemy pandas python-dotenv --quiet

### 2. Conexão com o banco (Supabase PostgreSQL)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '.env'), override=True)

usuario_db    = os.getenv('DB_USER')
senha_db      = os.getenv('DB_PASSWORD')
host_db       = os.getenv('DB_HOST')
porta_db      = int(os.getenv('DB_PORT'))
nome_banco_db = os.getenv('DB_NAME')

faltando = [k for k, v in {'DB_USER': usuario_db, 'DB_PASSWORD': senha_db, 'DB_HOST': host_db, 'DB_PORT': porta_db, 'DB_NAME': nome_banco_db}.items() if not v]
if faltando:
    raise ValueError(f'Variáveis ausentes no .env: {faltando}')

engine = create_engine(URL.create(
    drivername="postgresql+psycopg2",
    username=usuario_db,
    password=senha_db,
    host=host_db,
    port=porta_db,
    database=nome_banco_db,
    query={"client_encoding": "utf8"}
))

with engine.connect() as conn:
    print("Conectado:", conn.execute(text("SELECT version()")).scalar()[:50])

### 3. Análises

In [ ]:
# Análise 1 — Top 10 bairros por valor médio de avaliação
sql = """
SELECT
    l.bairro,
    COUNT(*)                             AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2)     AS valor_medio,
    ROUND(AVG(f.valor_m2_construido), 2) AS valor_medio_m2
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_localizacao l ON l.sk_localizacao = f.sk_localizacao
WHERE f.valor_avaliacao IS NOT NULL
GROUP BY l.bairro
ORDER BY valor_medio DESC
LIMIT 10
"""

with engine.connect() as conn:
    df_analise1 = pd.read_sql(text(sql), conn)

print("Top 10 bairros por valor médio de avaliação:")
df_analise1

In [ ]:
# Análise 2 — Evolução anual e trimestral do volume de transações
sql = """
SELECT
    t.ano,
    t.trimestre,
    COUNT(*)                         AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2) AS valor_medio_avaliacao
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_tempo t ON t.sk_tempo = f.sk_tempo
GROUP BY t.ano, t.trimestre
ORDER BY t.ano, t.trimestre
"""

with engine.connect() as conn:
    df_analise2 = pd.read_sql(text(sql), conn)

print("Evolução trimestral do volume de transações:")
df_analise2

In [ ]:
# Análise 3 — Valor médio por padrão de acabamento e tipo de imóvel
sql = """
SELECT
    c.tipo_imovel,
    c.padrao_acabamento,
    COUNT(*)                             AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2)     AS valor_medio,
    ROUND(AVG(f.valor_m2_construido), 2) AS valor_medio_m2
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_caracteristica c ON c.sk_caracteristica = f.sk_caracteristica
WHERE f.valor_m2_construido IS NOT NULL
GROUP BY c.tipo_imovel, c.padrao_acabamento
ORDER BY c.tipo_imovel, valor_medio_m2 DESC
"""

with engine.connect() as conn:
    df_analise3 = pd.read_sql(text(sql), conn)

print("Valor médio por padrão de acabamento e tipo de imóvel:")
df_analise3